# 12 — Resultados finais (TEA): consolidação dos notebooks 08, 09 e 10

Lê os resultados já gerados (tabelas longas e previsões OOF) e produz **a tabela e as figuras que vão para o TCC**. Nada é retreinado aqui.

**Regras de leitura**
- Os folds são validação cruzada agrupada por site, **não** um conjunto de teste independente. Escreva "validação cruzada por site", não "teste".
- Métrica principal: AUC-ROC. A "média entre folds" promedia as sementes dentro de cada fold e depois calcula média e desvio **entre os 5 folds**. A "AUC OOF" reúne as previsões fora do treino de todos os folds, com IC 95% por bootstrap de **sites**.
- Critério definido a priori: IC inferior > 0,5 ⇒ "acima do acaso". Não é prova de efeito nem de utilidade clínica. Não há teste de hipótese entre modelos (5 folds não sustentam Wilcoxon); diferenças entre modelos aparecem como intervalos, de forma exploratória.
- As referências de acaso (Dummy e rótulos embaralhados) entram na mesma tabela.
- Os limiares de decisão diferem por família: os baselines usam o corte padrão do modelo; deep learning e grafos usam um corte escolhido na validação interna. Compare modelos principalmente pela AUC.

In [ ]:
# 1. Projeto, módulo de avaliação e dataset (TEA / ABIDE I)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_raiz = Path.cwd().resolve()
for _c in (_raiz, *_raiz.parents):
    if (_c / 'src' / 'config.py').exists():
        _raiz = _c
        break
else:
    raise RuntimeError('Não encontrei src/config.py. Abra a pasta TCC_TEA_TDAH no VS Code.')
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src import avaliacao as av
criar_pastas()

NOME_BASE = 'TEA'
ROTULO_CASO = 'TEA'
ARQ_DATASET = PATHS['processed'] / 'dataset_abide_processado.pkl'
PASTA_RES = Path(PATHS['results'])
PASTA_MOD = Path(PATHS['models'])
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

# 2. Parâmetros
import json, glob
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, confusion_matrix

N_BOOT = 2000
if av.RAPIDO:
    N_BOOT = 200
    print('*** TCC_RAPIDO=1: teste curto ***')
pd.set_option('display.width', 200)
PASTA_FIG = Path(PATHS['figures']); PASTA_FIG.mkdir(parents=True, exist_ok=True)

d = av.carregar_dataset(ARQ_DATASET)
FAMILIA = {'Dummy_prior': 'Referência', 'Dummy_estratificado': 'Referência',
           'SVM_RBF': 'Clássico', 'Random_Forest': 'Clássico', 'XGBoost': 'Clássico',
           'CNN': 'Deep learning', 'ResNet50': 'Deep learning',
           'GCN': 'Grafo', 'GAT': 'Grafo', 'GraphSAGE': 'Grafo'}


def md_tabela(df, casas=3):
    """Tabela em Markdown (sem depender de pacotes extras)."""
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return '—' if np.isnan(v) else f'{v:.{casas}f}'
        return str(v)
    cab = '| ' + ' | '.join(['modelo'] + [str(c) for c in df.columns]) + ' |'
    sep = '|' + '|'.join(['---'] * (len(df.columns) + 1)) + '|'
    linhas = ['| ' + ' | '.join([str(i)] + [fmt(v) for v in r]) + ' |' for i, r in zip(df.index, df.values)]
    return '\n'.join([cab, sep] + linhas)

In [ ]:
# 3. Ler os resultados dos notebooks 08, 09 e 10
FONTES = [('08', '08_resultados_long.csv', '08_predicoes_oof.npz'),
          ('09', '09_resultados_long.csv', '09_predicoes_oof.npz'),
          ('10', '10_resultados_long.csv', '10_predicoes_oof.npz')]
longos, proba_oof, pred_oof, faltando = [], {}, {}, []
for nb_, csv, npz in FONTES:
    if not ((PASTA_RES / csv).exists() and (PASTA_RES / npz).exists()):
        faltando.append(nb_)
        continue
    L = pd.read_csv(PASTA_RES / csv)
    L['notebook'] = nb_
    z = np.load(PASTA_RES / npz, allow_pickle=True)
    if not (np.array_equal(z['y'], d['y']) and np.array_equal(z['fold_id'], d['fold_id'])
            and np.array_equal(z['ids'].astype(str), d['ids'])):
        raise ValueError(f'{npz} não corresponde ao dataset atual (participantes/folds diferentes). '
                         'Refaça os notebooks 08-10 com o mesmo notebook 07.')
    for m in L['modelo'].unique():
        if f'proba_{m.lower()}' in z.files:
            proba_oof[m], pred_oof[m] = z[f'proba_{m.lower()}'], z[f'pred_{m.lower()}']
    longos.append(L)
if faltando:
    print('AVISO: sem resultados dos notebooks', faltando, '-> a tabela final ficará incompleta.')
if not longos:
    raise FileNotFoundError('Nenhum resultado encontrado em ' + str(PASTA_RES))
df_long = pd.concat(longos, ignore_index=True)
if 'tipo' not in df_long.columns:
    df_long['tipo'] = 'externo'
print('Modelos encontrados:', sorted(df_long['modelo'].unique()))
print('Execuções por notebook:', df_long.groupby('notebook').size().to_dict())

In [ ]:
# 4. Tabela final
res = av.resumir_resultados(df_long, d, proba_oof)
pf, ic = res['por_fold'], res['auc_oof_ic'].set_index('modelo')
tab = pd.DataFrame({
    'família': [FAMILIA.get(m, 'Outro') for m in pf.index],
    'AUC média (folds)': pf[('auc_roc', 'mean')], 'AUC dp (folds)': pf[('auc_roc', 'std')],
    'AUC OOF': ic['auc'].reindex(pf.index), 'IC95 inf': ic['ic_inf'].reindex(pf.index), 'IC95 sup': ic['ic_sup'].reindex(pf.index),
    'Acc. balanceada': pf[('balanced_accuracy', 'mean')], 'Sensibilidade': pf[('recall', 'mean')],
    'Especificidade': pf[('specificity', 'mean')], 'F1': pf[('f1', 'mean')]})
tab['IC inferior > 0,5'] = np.where(tab['IC95 inf'].notna(), np.where(tab['IC95 inf'] > 0.5, 'sim', 'não'), '—')
colapsos = df_long[df_long['tipo'] == 'externo'].groupby('modelo')['classe_unica_prevista'].sum()
tab['execuções com 1 só classe prevista'] = colapsos.reindex(tab.index).fillna(0).astype(int)
tab = tab.sort_values('AUC média (folds)', ascending=False)
display(tab.round(3))

reais = tab[tab['família'] != 'Referência']
melhor = reais.index[0]
print(f'Melhor modelo pela AUC média entre folds: {melhor}')
print('Referências de acaso na mesma tabela: Dummy (AUC por fold = 0,5 por construção).')

In [ ]:
# 5. Figuras: AUC por fold, AUC OOF com IC, ROC
auc_fold = df_long[df_long['tipo'] == 'externo'].groupby(['modelo', 'fold'])['auc_roc'].mean().unstack().loc[tab.index]
fig, ax = plt.subplots(1, 2, figsize=(14, 0.55 * len(tab) + 2.5))
im = ax[0].imshow(auc_fold.values, cmap='RdYlGn', vmin=0.3, vmax=0.8, aspect='auto')
ax[0].set_xticks(range(auc_fold.shape[1])); ax[0].set_xticklabels([f'fold {c}' for c in auc_fold.columns])
ax[0].set_yticks(range(len(auc_fold))); ax[0].set_yticklabels(auc_fold.index)
for a in range(auc_fold.shape[0]):
    for b in range(auc_fold.shape[1]):
        ax[0].text(b, a, f'{auc_fold.values[a, b]:.2f}', ha='center', va='center', fontsize=9)
ax[0].set_title('AUC por fold externo (média das sementes)'); plt.colorbar(im, ax=ax[0], fraction=0.04)

t = tab[tab['IC95 inf'].notna()].iloc[::-1]
ax[1].errorbar(t['AUC OOF'], range(len(t)), xerr=[t['AUC OOF'] - t['IC95 inf'], t['IC95 sup'] - t['AUC OOF']], fmt='o', capsize=4)
ax[1].axvline(0.5, color='k', ls='--', lw=1); ax[1].set_yticks(range(len(t))); ax[1].set_yticklabels(t.index)
ax[1].set_xlabel('AUC OOF (IC 95%, bootstrap de sites)'); ax[1].set_title('AUC OOF e incerteza')
plt.tight_layout(); plt.savefig(PASTA_FIG / '12_auc_folds_e_ic.png', dpi=200); plt.show()

plt.figure(figsize=(6.5, 6))
for m in tab.index:
    if m in proba_oof and FAMILIA.get(m) != 'Referência':
        f_, t_, _ = roc_curve(d['y'], proba_oof[m])
        plt.plot(f_, t_, label=f"{m} ({tab.loc[m, 'AUC OOF']:.2f})")
plt.plot([0, 1], [0, 1], 'k--', alpha=.4)
plt.xlabel('1 − Especificidade'); plt.ylabel('Sensibilidade'); plt.legend(title='AUC OOF'); plt.title(f'Curvas ROC OOF — {NOME_BASE}')
plt.tight_layout(); plt.savefig(PASTA_FIG / '12_roc_oof.png', dpi=200); plt.show()
print('As curvas reúnem previsões de folds diferentes; sites com prevalências distintas podem deslocar a curva.')

In [ ]:
# 6. Matrizes de confusão (previsões OOF) dos 3 melhores modelos e desempenho por site do melhor
top3 = list(reais.index[:3])
fig, ax = plt.subplots(1, len(top3), figsize=(4.6 * len(top3), 4.2), squeeze=False)
for a, m in zip(ax[0], top3):
    cm = confusion_matrix(d['y'], pred_oof[m], labels=[0, 1])
    pct = cm / cm.sum(1, keepdims=True)
    a.imshow(pct, cmap='Blues', vmin=0, vmax=1)
    for r in range(2):
        for c in range(2):
            a.text(c, r, f'{cm[r, c]}\n({pct[r, c]:.0%})', ha='center', va='center', color='white' if pct[r, c] > .5 else 'black')
    a.set_xticks([0, 1]); a.set_xticklabels(['Controle', ROTULO_CASO]); a.set_yticks([0, 1]); a.set_yticklabels(['Controle', ROTULO_CASO])
    a.set_xlabel('Previsto'); a.set_ylabel('Real'); a.set_title(m)
plt.tight_layout(); plt.savefig(PASTA_FIG / '12_matrizes_confusao.png', dpi=200); plt.show()

tab_site = av.tabela_por_site(d, proba_oof[melhor], pred_oof[melhor])
print(f'DESEMPENHO POR SITE — {melhor} (AUC só existe em sites com as duas classes)')
display(tab_site.round(3))
print('Atenção: sites com uma só classe misturam efeito de site e diagnóstico; leia o desempenho global com cautela.')

In [ ]:
# 7. Comparação com incerteza: melhor modelo x os demais (bootstrap de sites, exploratório)
def dif_auc_boot(y, pa, pb, sites, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    nomes = np.unique(sites); idx_s = {s: np.where(sites == s)[0] for s in nomes}
    difs = []
    for _ in range(n_boot):
        idx = np.concatenate([idx_s[s] for s in rng.choice(nomes, size=len(nomes), replace=True)])
        if len(np.unique(y[idx])) < 2:
            continue
        difs.append(av.roc_auc_score(y[idx], pa[idx]) - av.roc_auc_score(y[idx], pb[idx]))
    difs = np.asarray(difs)
    return dict(dif=float(av.roc_auc_score(y, pa) - av.roc_auc_score(y, pb)),
                ic_inf=float(np.percentile(difs, 2.5)), ic_sup=float(np.percentile(difs, 97.5)))

linhas = []
for m in reais.index[1:]:
    linhas.append(dict(comparacao=f'{melhor} − {m}', **dif_auc_boot(d['y'], proba_oof[melhor], proba_oof[m], d['sites'])))
tab_comp = pd.DataFrame(linhas)
display(tab_comp.round(3))
print('Se o IC da diferença inclui 0, os dados não separam os dois modelos. Não é um teste de hipótese; é descrição da incerteza.')

In [ ]:
# 8. Controles, sensibilidade e diagnóstico de colapso
arq_perm = PASTA_RES / '08_teste_rotulos_embaralhados.csv'
if arq_perm.exists():
    dp = pd.read_csv(arq_perm)
    print('ROTULOS EMBARALHADOS (SVM, AUC no fold de teste): média = %.3f, dp = %.3f, máx = %.3f (n = %d)' %
          (dp['auc'].mean(), dp['auc'].std(), dp['auc'].max(), len(dp)))
    print('  -> deve ficar perto de 0,5. Média claramente acima indica vazamento ou erro de pipeline.' if dp['auc'].mean() > 0.55
          else '  -> compatível com o acaso.')
else:
    print('08_teste_rotulos_embaralhados.csv não encontrado.')

print('\nSENSIBILIDADE (notebook 08 rodado com EXCLUIR_SITES; arquivos com sufixo _sem_...)')
achou = False
for arq in sorted(glob.glob(str(PASTA_RES / '08_resultados_long_sem_*.csv'))):
    achou = True
    ls = pd.read_csv(arq)
    r = ls[ls['tipo'] == 'externo'].groupby(['modelo', 'fold'])['auc_roc'].mean().groupby('modelo').agg(['mean', 'std']).round(3)
    print('Arquivo:', Path(arq).name); display(r)
if not achou:
    print('Nenhuma análise de sensibilidade encontrada. (TDAH: rodar o 08 com EXCLUIR_SITES = [Pittsburgh, WashU].)')

print('\nCOLAPSO E AMPLITUDE DA AUC POR FOLD')
dg = df_long[df_long['tipo'] == 'externo'].groupby('modelo').agg(
    execucoes=('auc_roc', 'size'), colapsos=('classe_unica_prevista', 'sum'),
    auc_min=('auc_roc', 'min'), auc_max=('auc_roc', 'max')).round(3)
display(dg)

In [ ]:
# 9. Texto-base e arquivos finais
b = tab.loc[melhor]
if b['IC95 inf'] > 0.5:
    frase_ic = ('O limite inferior do IC ficou acima de 0,5, o que atende ao critério definido a priori; '
                'isso não constitui prova de efeito nem de utilidade clínica.')
else:
    frase_ic = ('O IC incluiu 0,5; os resultados não permitem afirmar desempenho acima do acaso.')
refs = tab[tab['família'] == 'Referência']
texto = (f'Na base {NOME_BASE}, o melhor desempenho médio entre folds foi do modelo {melhor} '
         f'(AUC = {b["AUC média (folds)"]:.3f} ± {b["AUC dp (folds)"]:.3f}; AUC OOF = {b["AUC OOF"]:.3f}, '
         f'IC 95% por bootstrap de sites [{b["IC95 inf"]:.3f}, {b["IC95 sup"]:.3f}]). {frase_ic} '
         f'Acurácia balanceada média de {b["Acc. balanceada"]:.3f}, sensibilidade {b["Sensibilidade"]:.3f} e especificidade {b["Especificidade"]:.3f}. '
         'Os resultados vêm de validação cruzada de 5 folds agrupada por site, não de um conjunto de teste independente; '
         'não foram comparados com o diagnóstico clínico.')
print(texto)

tab.to_csv(PASTA_RES / '12_tabela_final.csv')
tab_comp.to_csv(PASTA_RES / '12_comparacao_com_melhor.csv', index=False)
df_long.to_csv(PASTA_RES / '12_resultados_consolidados_long.csv', index=False)
with open(PASTA_RES / '12_tabela_final.md', 'w', encoding='utf-8') as fh:
    fh.write(md_tabela(tab.round(3)) + '\n\n' + texto + '\n')
print('\nSalvo em', PASTA_RES, '(12_tabela_final.csv / .md, 12_comparacao_com_melhor.csv) e figuras em', PASTA_FIG)

## Antes de escrever o capítulo de resultados
- Confira se as tabelas dos notebooks 08, 09 e 10 vieram da **mesma versão** do notebook 07 (o carregamento acima recusa arquivos de outro conjunto de participantes).
- Descreva sempre: 5 folds por site, médias entre folds, IC por bootstrap de sites, critério a priori e ausência de teste independente.
- Resultados perto do acaso são um resultado: relate-os junto com as referências (Dummy, rótulos embaralhados) e com os limites do dado (sites, protocolo, movimento).
- Não compare com o diagnóstico clínico: este estudo não mediu isso.